# LangChain Tool Execution & Return Values

Each section is a **complete `create_agent` example**. Run the sections in order.

## Shared setup

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.types import Command

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Return a string

Use a string when the tool returns plain text for the model to read. The model then generates the final response.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is currently sunny in {city}."

agent = create_agent(
    model=model,
    tools=[get_weather],
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the weather in Pune?"}
    ]
})

print(result["messages"][-1].content)

**Behavior:** The string becomes a `ToolMessage`. The agent sends the tool result back to the model, which creates the final answer. No graph state field is changed directly.

## 2. Return an object

Use a dictionary when the tool returns structured data that the model should inspect.

In [ ]:
@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

agent = create_agent(
    model=model,
    tools=[get_weather_data],
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Get the weather data for Pune and explain it briefly."}
    ]
})

print(result["messages"][-1].content)

**Behavior:** The dictionary is serialized as tool output. The model can reason over its fields. Returning an object does not directly update graph state.

## 3. Return multimodal content

A tool can return text and image content blocks. The model must support the modality being returned. This example uses a public sample image URL; replace it with an accessible image URL for your use case.

In [ ]:
@tool
def capture_screenshot() -> list[dict]:
    """Return a screenshot image with a short description."""
    return [
        {"type": "text", "text": "Screenshot of the current page:"},
        {
            "type": "image",
            "url": "https://example.com/page.png",
        },
    ]

agent = create_agent(
    model=model,
    tools=[capture_screenshot],
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Capture the page screenshot and tell me what you received."}
    ]
})

print(result["messages"][-1].content)

**Behavior:** The tool result is represented as multimodal content in a `ToolMessage`. Use `message.content_blocks` to inspect normalized blocks. The sample URL is illustrative and may need replacing with a real image.

## 4. Return a Command

Use `Command` when the tool must update graph state. For a command targeting the current graph, include a `ToolMessage` whose `tool_call_id` matches `runtime.tool_call_id`.

In [ ]:
@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Set the user's preferred response language in agent state."""
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Language set to {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

agent = create_agent(
    model=model,
    tools=[set_language],
    state_schema=None,
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Set my preferred language to Marathi."}
    ]
})

print("Final response:", result["messages"][-1].content)
print("Preferred language in state:", result.get("preferred_language"))

**Behavior:** `Command.update` writes `preferred_language` into the agent state. The matching `ToolMessage` keeps the message history valid. The updated state is available to subsequent steps in the same run.

## 5. Return directly from a tool

Set `return_direct=True` when the tool output is already the complete answer. The agent exits without another model call after the tool execution.

In [ ]:
@tool(return_direct=True)
def fetch_order_status(order_id: str) -> str:
    """Fetch the current status of a customer order."""
    return f"Order {order_id} is shipped and will arrive in 2 days."

agent = create_agent(
    model=model,
    tools=[fetch_order_status],
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the status of order 12345?"}
    ]
})

print(result["messages"][-1].content)

**Behavior:**
- The tool executes and its output is recorded as a `ToolMessage`.
- The agent returns the tool output without another LLM call.
- Use this when the tool output is already user-ready.
- If parallel tools run in one step, the agent exits only when every tool in that batch has `return_direct=True`. If the batch mixes direct and normal tools, the model receives the results and continues.

## 6. Return a Command with `return_direct=True`

This combines both behaviors: update state and then exit the agent loop. The `ToolMessage` is still required when the command targets the current graph.

In [ ]:
@tool(return_direct=True)
def fetch_and_store_order(
    order_id: str,
    runtime: ToolRuntime,
) -> Command:
    """Fetch order status and save it in agent state."""
    status = f"Order {order_id} is shipped and will arrive in 2 days."

    return Command(
        update={
            "last_order_status": status,
            "messages": [
                ToolMessage(
                    content=status,
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

agent = create_agent(
    model=model,
    tools=[fetch_and_store_order],
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Fetch and store the status of order 12345."}
    ]
})

print("Tool result:", result["messages"][-1].content)
print("Stored state:", result.get("last_order_status"))

**Parent graph note:** If a tool returns `Command(graph=Command.PARENT, ...)`, execution updates the parent graph. The current-graph `ToolMessage` requirement described above does not apply because execution leaves the current graph.

## Quick recap

| Return style | Purpose | Direct state update? |
|---|---|---:|
| `str` | Plain-text result for the model | No |
| `dict` / object | Structured result for the model | No |
| Multimodal content | Text, image, audio, or video blocks | No |
| `Command` | Update graph state | Yes |
| `return_direct=True` | Stop the agent loop after tool execution | No, by itself |
| `Command` + `return_direct=True` | Update state and stop the agent loop | Yes |

**Remember:** `return_direct` controls the agent loop. `Command` controls graph-state updates.